In [2]:
from train import data
import dspy
print(data)

[('Show marketing calendar for June', 'rag'), ('What does docs/marketing_calendar.md say about beverages?', 'rag'), ('Find policies in docs/product_policy.md', 'rag'), ('Search product_policy return window for dairy', 'rag'), ('Retrieve marketing_calendar::chunk0 about Winter Classics', 'rag'), ('Look up KPI definitions in docs/kpi_definitions.md', 'rag'), ('Give me documentation on categories in docs/catalog.md', 'rag'), ('Find notes in marketing_calendar for 1997 Winter', 'rag'), ('Which documents mention Beverages?', 'rag'), ('Show doc chunks referencing AOV', 'rag'), ('Total revenue between 1997-06-01 and 1997-06-30', 'sql'), ('How many orders in June 1997?', 'sql'), ('AOV for Beverages category last month', 'sql'), ('Write SQL to compute gross margin by category', 'sql'), ('Orders grouped by customer region for 1997-12', 'sql'), ('Return OrderID, Quantity where discount > 0.1', 'sql'), ('Show top 5 products by revenue', 'sql'), ('Run SQL: SELECT SUM(UnitPrice*Quantity) FROM OrderD

In [7]:
len(data)

29

In [13]:
import random
random.shuffle(data)

split = int(len(data) * 0.8)
train_data = data[:split]
test_data  = data[split:]

trainset = [
    dspy.Example(question=q, mode=m).with_inputs("question")
    for q, m in train_data
]
testset = [
    dspy.Example(question=q, mode=m).with_inputs("question")
    for q, m in test_data
]

In [17]:
optimizer = dspy.MIPROv2(
    metric=lambda gold, pred: gold.mode == pred.mode,
    max_bootstrapped_demos=10,     # use more demos since you have 29 examples
    max_labeled_demos=20,          # optional (safe)
)

from agent.dspy_signatures import RouterModule
router = RouterModule()
optimized_router = optimizer.compile(
    module=router,
    trainset=trainset
)

TypeError: MIPROv2.compile() got an unexpected keyword argument 'module'

In [16]:
from agent.dspy_signatures import RouterModule
import pickle
router = RouterModule()

optimized_router = optimizer.compile(
    router,
    trainset
)

with open("optimized_router.pkl", "wb") as f:
    pickle.dump(optimized_router, f)

TypeError: MIPROv2.compile() takes 2 positional arguments but 3 were given

In [ ]:
def evaluate(module, dataset):
    correct = 0
    for ex in dataset:
        pred = module(question=ex.question)["mode"]
        if pred == ex.mode:
            correct += 1
    return correct / len(dataset)

print("Train accuracy:", evaluate(optimized_router, trainset))
print("Test accuracy:",  evaluate(optimized_router, testset))


In [ ]:
def evaluate(module, dataset):
    correct = 0
    for ex in dataset:
        pred = module(question=ex.question)["mode"]
        if pred == ex.mode:
            correct += 1
    return correct / len(dataset)

print("Train accuracy:", evaluate(optimized_router, trainset))
print("Test accuracy:",  evaluate(optimized_router, testset))
